# ML-00 baseline analysis from approved ML inputs

`ml_inputs/ml_00_test/run_00`에 사람이 승인한 train/val/test parquet와 feature mask를 입력으로 사용해 ML-00 XGBoost baseline 분석을 수행한다.

## 실행 범위

- 이 노트북은 feature 생성이나 `fb_outputs -> ml_inputs` 복사를 수행하지 않는다.
- 입력은 `ml_inputs`에 배치된 승인본만 사용한다.
- 기본값은 `RUN_TRAIN_AND_VALIDATION=False`라서 Run All 시 입력 계약, feature mask, split_summary, schema만 확인한다.
- split label 분포는 승인 입력의 `split_summary.csv`를 기준으로 확인하되, test label 분포는 `SHOW_TEST_LABEL_DISTRIBUTION=True`일 때만 표시한다.
- sample 결과는 smoke/debug 전용이며 성능 주장에 사용하지 않는다.
- final test는 full model과 validation threshold 확정 후 1회만 실행한다. 기본값은 `RUN_FINAL_TEST=False`다.

# 00_경로 및 환경설정

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

def display_status_table(df: pd.DataFrame) -> None:
    """값이 잘리지 않도록 pandas 표시 옵션을 임시 적용한다."""
    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_columns", None,
        "display.width", None,
    ):
        display(df)


# ============================================================
# 0.1 Runtime Settings
# ============================================================
# 실행 환경: ML 담당자 기준
# - Kernel          : Local WSL
# - Code repo       : local Git repo
# - Input artifacts : local Git repo ml/ml-00_baseline/ml_inputs/
# - Output artifacts: local Git repo ml/ml-00_baseline/ml_outputs/
SEED = 42

# Root Paths: 다른 환경에서 실행할 경우 보통 이 경로만 수정한다.
LOCAL_REPO_ROOT = Path("/mnt/d/AML_git/Graph_AML").resolve()

# Input/Output Paths
ML_00_BASE_DIR = LOCAL_REPO_ROOT / "ml" / "ml-00_baseline"
ML_00_ML_INPUTS_DIR = ML_00_BASE_DIR / "ml_inputs"
ML_00_ML_OUTPUTS_DIR = ML_00_BASE_DIR / "ml_outputs"

# Module Code Paths: local Git repo에서 train_val_test 모듈을 import한다.
ML_00_MODULE_TVT_DIR = ML_00_BASE_DIR / "train_val_test"


# ============================================================
# 0.2 Path Validation
# ============================================================
def require_dir(path: Path, name: str) -> None:
    """필수 디렉터리가 없으면 시작 단계에서 중단한다."""
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"{name} is not a directory: {path}")


REQUIRED_DIRS = {
    "LOCAL_REPO_ROOT": LOCAL_REPO_ROOT,
    "ML_00_BASE_DIR": ML_00_BASE_DIR,
    "ML_00_ML_INPUTS_DIR": ML_00_ML_INPUTS_DIR,
    "ML_00_ML_OUTPUTS_DIR": ML_00_ML_OUTPUTS_DIR,
    "ML_00_MODULE_TVT_DIR": ML_00_MODULE_TVT_DIR,
}
for name, path in REQUIRED_DIRS.items():
    require_dir(path, name)


# ============================================================
# 0.3 Import Path Setup
# ============================================================
# local train_val_test/ml_00_ml_*.py 모듈을 우선 import한다.
IMPORT_DIRS = [str(ML_00_MODULE_TVT_DIR)]
IMPORT_MODULES = (
    "ml_00_ml_utils",
    "ml_00_ml_io",
    "ml_00_ml_preview",
    "ml_00_ml_train",
    "ml_00_ml_val",
    "ml_00_ml_test",
)

# 중복 경로를 제거한 뒤 맨 앞에 삽입해 import 우선순서를 고정한다.
sys.path = [path for path in sys.path if path not in IMPORT_DIRS]
sys.path[:0] = IMPORT_DIRS

# 노트북 재실행 시 수정된 local 모듈을 다시 읽도록 import cache를 제거한다.
for module_name in IMPORT_MODULES:
    sys.modules.pop(module_name, None)

import ml_00_ml_utils
import ml_00_ml_io
import ml_00_ml_preview
import ml_00_ml_train
import ml_00_ml_val
import ml_00_ml_test

ml_00_ml_utils.set_seed(SEED, use_torch=False)

PROJECT_ROOT = LOCAL_REPO_ROOT
ML_UTILS_PATH = Path(ml_00_ml_utils.__file__).resolve()

# ============================================================
# 0.4 Configuration Summary
# ============================================================
print("=" * 60)
print(f"SEED                           : {SEED}")
print("-" * 60)
print(f"LOCAL_REPO_ROOT                : {LOCAL_REPO_ROOT}")
print(f"PROJECT_ROOT                   : {PROJECT_ROOT}")
print("-" * 60)
print(f"ML_00_BASE_DIR                 : {ML_00_BASE_DIR}")
print(f"ML_00_ML_INPUTS_DIR            : {ML_00_ML_INPUTS_DIR}")
print(f"ML_00_ML_OUTPUTS_DIR           : {ML_00_ML_OUTPUTS_DIR}")
print("-" * 60)
print(f"ML_00_MODULE_TVT_DIR           : {ML_00_MODULE_TVT_DIR}")
print("=" * 60 + "\n")


SEED                           : 42
------------------------------------------------------------
LOCAL_REPO_ROOT                : /mnt/d/AML_git/Graph_AML
PROJECT_ROOT                   : /mnt/d/AML_git/Graph_AML
------------------------------------------------------------
ML_00_BASE_DIR                 : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline
ML_00_ML_INPUTS_DIR            : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs
ML_00_ML_OUTPUTS_DIR           : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs
------------------------------------------------------------
ML_00_MODULE_TVT_DIR           : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/train_val_test



# 01_실행 설정

In [2]:
# ============================================================
# 실행 예시
# ============================================================
# 예시 1. 입력 검증만 실행
# - 입력 파일, feature mask, split_summary, schema만 확인
# - 학습/validation/final test는 실행하지 않음
#   MODEL_RUN_ID = "xgb_default_00"
#   RUN_TRAIN_AND_VALIDATION = False
#   SAMPLE_ROWS = 100_000
#   RUN_FINAL_TEST = False
#   USE_NOTEBOOK_XGB_PARAMS = True
#
# 예시 2. Full train/validation만 실행
# - 전체 train/val split으로 모델 학습 + validation threshold 선택
# - final test는 아직 실행하지 않음
#   MODEL_RUN_ID = "xgb_default_00"
#   RUN_TRAIN_AND_VALIDATION = True
#   SAMPLE_ROWS = None
#   RUN_FINAL_TEST = False
#   USE_NOTEBOOK_XGB_PARAMS = False
#
# 예시 3. 기존 full artifact로 final test만 실행
# - 같은 OUT_DIR(= .../full/MODEL_RUN_ID)에 model.pkl, feature_columns.json, train_summary.json, threshold.json이 있을 때만 실행
# - train/validation은 다시 돌리지 않고 final test만 수행
#   MODEL_RUN_ID = "xgb_default_00"
#   RUN_TRAIN_AND_VALIDATION = False
#   SAMPLE_ROWS = None
#   RUN_FINAL_TEST = True
#   USE_NOTEBOOK_XGB_PARAMS = False
#
# 주의:
# - EXPORT_EXPERIMENT_ID/RUN_ID는 fb 노트북과 같은 승인 입력 묶음 식별자다.
# - MODEL_RUN_ID는 같은 입력으로 반복하는 모델 학습/검증 시도 식별자다.
# - RUN_KIND는 sample/full 구분용이다.
# - train/val/test는 하위 폴더가 아니라 파일명으로 구분한다.
# - OVERWRITE_POLICY가 False이면 기존 산출물이 있을 때 덮어쓰지 않고 중단한다.
# ============================================================
# 산출물 구분
# ============================================================
# model.pkl                    -> train 결과
# feature_columns.json          -> train 결과
# train_summary.json            -> train 결과 + runtime/memory_mb/resource/xgboost diagnostics
# feature_importance.csv         -> train 결과, booster importance
#
# threshold.json                -> validation 결과
# metrics_val.json              -> validation 결과 + score/memory_mb/resource profile
# confusion_matrix_val.csv      -> validation 결과
#
# metrics_test.json             -> final test 결과 + score/memory_mb/resource profile
# confusion_matrix_test.csv     -> final test 결과

In [3]:
# ============================================================
# 1.1 Run identifiers
# ============================================================
EXPORT_EXPERIMENT_ID = "ml_00_test"
RUN_ID = "run_00_no_hour"

# MODEL_RUN_ID: train_val_test 노트북에서만 쓰는 모델 시도 식별자
# - 같은 승인 입력(EXPORT_EXPERIMENT_ID/RUN_ID)으로 파라미터, seed, threshold 전략을 바꿔 재시도할 때 변경
MODEL_RUN_ID = "default_00_refactoring_no_hour"
ARTIFACT_PREFIX = f"{EXPORT_EXPERIMENT_ID}__{RUN_ID}"
LABEL_COL = "label" # 정답 라벨 컬럼명

# ============================================================
# 1.2 Execution switches
# ============================================================
# 사용자가 주로 수정할 실행 모드 스위치: 기본값은 "입력 검증 + feature 목록 확인"까지만 수행하고 학습은 실행하지 않음
# - False: 입력 파일, feature mask, schema, split_summary QA만 확인
# - True : XGBoost train + validation threshold selection 실행
RUN_TRAIN_AND_VALIDATION = True

# sample smoke/debug 전용 row 제한
# - 100_000: train/val parquet 앞쪽 100,000행만 사용
# - None: 전체 train/val split 사용
SAMPLE_ROWS: int | None = None

# final test 실행 여부 : 기본값 False 유지
# - True는 full train/validation과 threshold가 확정된 뒤 1회만 사용
# - SAMPLE_ROWS가 None이 아니면 final test는 차단됨
RUN_FINAL_TEST = False

# XGBoost 설정 사용 방식
# - True: 아래 XGB_PARAMS 값으로 ml_00_ml_train.XGBTrainConfig 기본값을 덮어씀
# - False: 모듈의 XGBTrainConfig 기본값을 그대로 사용
USE_NOTEBOOK_XGB_PARAMS = False


# test label 분포 표시 여부: final test 전 기본 숨김
# - False: test의 label count/positive_rate를 화면과 반환 summary에서 숨김
# - True: 사용자가 명시 승인한 경우에만 test label 분포 표시
SHOW_TEST_LABEL_DISTRIBUTION = False

# ============================================================
# 1.3 Execution policy and model settings
# RUN_TRAIN_AND_VALIDATION=True일 때만 실제로 사용되는 학습 설정
# ============================================================
# 기존 output artifact overwrite 방지 정책 : 
# - False: 기존 산출물이 있으면 실행 중단
# - True: 기존 산출물을 덮어씀
OVERWRITE_POLICY = {
    "train": False,
    "val": False,
    "test": False,
}

# XGBoost 실행별 override 설정 : 현재 값은 sample smoke run 기준 경량 설정
# - USE_NOTEBOOK_XGB_PARAMS=True일 때만 ml_00_ml_train.XGBTrainConfig에 전달됨
# - full run에서는 n_estimators, early_stopping_rounds, n_jobs 재검토 필요
XGB_PARAMS = {
    "n_estimators": 50,          # tree 개수 상한. sample 실행이라 낮게 설정
    "max_depth": 4,              # tree 최대 깊이
    "learning_rate": 0.05,       # 각 tree 반영 강도
    "subsample": 0.9,            # row subsampling 비율
    "colsample_bytree": 0.9,     # feature subsampling 비율
    "min_child_weight": 1.0,     # leaf 생성 최소 조건
    "reg_lambda": 1.0,           # L2 정규화
    "reg_alpha": 0.0,            # L1 정규화
    "gamma": 0.0,                # split 추가 최소 loss 감소량
    "early_stopping_rounds": 10, # validation 성능 개선 없을 때 조기 중단 기준
    "n_jobs": 1,                 # CPU thread 수. sample 실행 안정성을 위해 1로 제한
}

# validation threshold 선택 정책
# - max_f1: validation set에서 F1이 최대인 threshold 자동 선택
# - manual: MANUAL_THRESHOLD 값을 고정 threshold로 사용
# 예: THRESHOLD_STRATEGY = "manual", MANUAL_THRESHOLD = 0.42
THRESHOLD_STRATEGY = "max_f1"
MANUAL_THRESHOLD = None

# ============================================================
# 1.4 Approved ML input file and paths
# ============================================================
ML_INPUT_DIR = ML_00_ML_INPUTS_DIR / EXPORT_EXPERIMENT_ID / RUN_ID
ML_OUTPUT_DIR = ML_00_ML_OUTPUTS_DIR / EXPORT_EXPERIMENT_ID / RUN_ID

# 승인 ML 입력 파일 경로:  feature_columns_approve.csv의 used_in_ml="TRUE"인 컬럼만 모델 feature로 사용됨
ML_INPUT_TRAIN_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_train.parquet"
ML_INPUT_VAL_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_val.parquet"
ML_INPUT_TEST_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_Xy_test.parquet"
ML_INPUT_FEATURE_COLUMNS_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_ml_feature_columns_approve.csv"
ML_INPUT_SPLIT_SUMMARY_PATH = ML_INPUT_DIR / f"{ARTIFACT_PREFIX}_split_summary.csv"

# INPUT_PATHS: train/val/test/feature mask 경로를 하나의 객체로 묶어 후속 함수에 전달
INPUT_PATHS = ml_00_ml_io.InputPaths(
    train_path=ML_INPUT_TRAIN_PATH,
    val_path=ML_INPUT_VAL_PATH,
    test_path=ML_INPUT_TEST_PATH,
    feature_columns_path=ML_INPUT_FEATURE_COLUMNS_PATH,
)

# ============================================================
# 1.5 Output paths
# ============================================================
# RUN_KIND: sample 실행과 full 실행 산출물이 섞이지 않도록 output 하위 폴더를 분리
# - SAMPLE_ROWS=100_000이면 sample_100000 / SAMPLE_ROWS=None이면 full
RUN_KIND = f"sample_{SAMPLE_ROWS}" if SAMPLE_ROWS is not None else "full"

# OUT_DIR: 실제 학습/validation/final test artifact가 저장되는 최종 폴더
# - EXPORT_EXPERIMENT_ID/RUN_ID는 승인 입력 묶음을 가리킨다.
# - MODEL_RUN_ID는 같은 승인 입력 안에서 모델 시도를 분리한다.
# - 예: ml_outputs/ml_00_test/run_00/full/xgb_default_001
OUT_DIR = ML_OUTPUT_DIR / RUN_KIND / MODEL_RUN_ID
OUT_DIR = Path(OUT_DIR).expanduser().resolve()

# ============================================================
# 1.6 Configuration preview
# ============================================================
print("=" * 80)
print("EXPORT_EXPERIMENT_ID          :", EXPORT_EXPERIMENT_ID)
print("RUN_ID                        :", RUN_ID)
print("MODEL_RUN_ID                  :", MODEL_RUN_ID)
print("ARTIFACT_PREFIX               :", ARTIFACT_PREFIX)
print("LABEL_COL                     :", LABEL_COL)
print("-" * 80)
print("RUN_KIND                      :", RUN_KIND)
print("RUN_TRAIN_AND_VALIDATION      :", RUN_TRAIN_AND_VALIDATION)
print("SAMPLE_ROWS                   :", SAMPLE_ROWS)
print("RUN_FINAL_TEST                :", RUN_FINAL_TEST)
print("SHOW_TEST_LABEL_DISTRIBUTION  :", SHOW_TEST_LABEL_DISTRIBUTION)
print("USE_NOTEBOOK_XGB_PARAMS       :", USE_NOTEBOOK_XGB_PARAMS)
print("-" * 80)
print("ML_INPUT_DIR                  :", ML_INPUT_DIR)
print("ML_INPUT_TRAIN_PATH           :", ML_INPUT_TRAIN_PATH)
print("ML_INPUT_VAL_PATH             :", ML_INPUT_VAL_PATH)
print("ML_INPUT_TEST_PATH            :", ML_INPUT_TEST_PATH)
print("ML_INPUT_FEATURE_COLUMNS_PATH :", ML_INPUT_FEATURE_COLUMNS_PATH)
print("ML_INPUT_SPLIT_SUMMARY_PATH   :", ML_INPUT_SPLIT_SUMMARY_PATH)
print("-" * 80)
print("ML_OUTPUT_DIR                 :", ML_OUTPUT_DIR)
print("OUT_DIR                       :", OUT_DIR)
print("=" * 80)


EXPORT_EXPERIMENT_ID          : ml_00_test
RUN_ID                        : run_00_no_hour
MODEL_RUN_ID                  : default_00_refactoring_no_hour
ARTIFACT_PREFIX               : ml_00_test__run_00_no_hour
LABEL_COL                     : label
--------------------------------------------------------------------------------
RUN_KIND                      : full
RUN_TRAIN_AND_VALIDATION      : True
SAMPLE_ROWS                   : None
RUN_FINAL_TEST                : False
SHOW_TEST_LABEL_DISTRIBUTION  : False
USE_NOTEBOOK_XGB_PARAMS       : False
--------------------------------------------------------------------------------
ML_INPUT_DIR                  : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour
ML_INPUT_TRAIN_PATH           : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_Xy_train.parquet
ML_INPUT_VAL_PATH             : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/r

# 02_입력 계약 검증 함수 정의

In [4]:
def artifact_file_status(name: str, path: Path) -> dict:
    """
    입력/출력 artifact의 존재 여부와 수정 시각을 표 형태로 확인하기 위한 row dict를 만든다.
    사용 목적
    --------
    - 노트북 마지막 단계에서 입력 파일과 output artifact 상태를 한눈에 확인한다.
    - 학습을 실행하지 않은 상태에서는 output artifact가 없어야 정상일 수 있다.
    - 기존 artifact가 이미 있는지 확인해 overwrite 차단 원인을 빠르게 파악한다.
    - path가 파일이 아니면 stat()을 호출하지 않고 exists/is_file 상태만 반환한다.
    """
    path = Path(path)
    is_file = path.is_file()
    
    # 파일이 없거나 디렉터리인 경우: 수정 시각을 계산하지 않고 상태만 기록한다.
    if not is_file:
        return {
            "name": name,
            "exists": path.exists(),
            "is_file": False,
            "modified_at": None,
            "path": str(path),
        }
        
    # 실제 파일인 경우에만 수정 시각을 기록한다.
    stat = path.stat()
    return {
        "name": name,
        "exists": True,
        "is_file": True,
        "modified_at": pd.Timestamp.fromtimestamp(stat.st_mtime),
        "path": str(path),
    }

print("functions loaded")

functions loaded


# 03_모델 입력 feature 목록 최종 확인

In [5]:
def preview_ml_inputs(paths: ml_00_ml_io.InputPaths) -> tuple[list[str], pd.DataFrame]:
    """
    승인된 ML 입력 묶음을 학습 전에 검증하고, 사람이 확인해야 할 핵심 표를 출력한다.
    데이터 흐름
    --------
    1. paths에 지정된 train/val/test parquet와 feature approve CSV를 입력으로 받는다.
    2. ml_00_ml_preview.preview_ml_inputs()가 파일 존재, feature mask, split_summary, parquet schema를 검증한다.
    3. 노트북에서는 검증 결과를 사람이 확인할 수 있도록 feature 목록, split 요약, schema 상태를 출력한다.
    4. main()에서 후속 train/validation 단계가 재사용할 feature_columns와 split_summary를 반환한다.
    주의
    ----
    - test label 분포는 SHOW_TEST_LABEL_DISTRIBUTION=True일 때만 표시한다.
    - 실제 검증 로직은 ml_00_ml_preview.py에 두고, 노트북은 실행 흐름과 표시만 담당한다.
    """
    # 1. 표준 preview 모듈에서 승인 입력 계약을 먼저 검증한다.
    # 검증 항목:
    # - train/val/test parquet와 approve CSV 존재 여부
    # - used_in_ml="TRUE" feature 목록 검증
    # - split_summary.csv의 split/row/label count 정합성
    # - parquet schema에 모델 feature와 필수 운영 컬럼 존재 여부
    preview = ml_00_ml_preview.preview_ml_inputs(
        paths,
        split_summary_path=ML_INPUT_SPLIT_SUMMARY_PATH,
        project_root=PROJECT_ROOT,
        label_col=LABEL_COL,
        show_test_label_distribution=SHOW_TEST_LABEL_DISTRIBUTION,
    )
    
    # 2. 현재 노트북이 실제로 읽을 입력 파일 경로를 출력한다. 
    ml_00_ml_io.print_input_paths(paths)
    
    # 3. 모델에 들어갈 feature mask 승인본을 표시한다.selected_features_display는 approve CSV 중 used_in_ml="TRUE"인 feature만 담는다.
    print("model_feature_columns_path:", paths.feature_columns_path)
    print("model_feature_count:", len(preview.feature_columns))
    print("모델 입력 feature 최종 확인표")
    display_status_table(preview.selected_features_display)
    
    # 4. feature 순서와 hash를 출력한다.
    # feature_columns_hash는 같은 feature라도 순서가 바뀌면 달라지므로 train/validation/test artifact provenance 확인에 사용된다.
    print("feature_count:", len(preview.feature_columns))
    print("feature_columns_hash:", preview.feature_columns_hash)
    print("feature_columns:", preview.feature_columns)
    
    # 5. split_summary.csv를 기준으로 train/val/test row 수와 label 분포를 표시한다.
    # test label 분포는 최종 평가 전 정보 노출을 줄이기 위해 기본 숨김 처리한다.
    print("split_summary_path:", ML_INPUT_SPLIT_SUMMARY_PATH)
    if SHOW_TEST_LABEL_DISTRIBUTION:
        print("전처리/feature build split label 분포 요약")
    else:
        print("전처리/feature build split 요약 (test label 분포 숨김)")
        print("SHOW_TEST_LABEL_DISTRIBUTION=True일 때만 test label 분포를 표시합니다.")
    display_status_table(preview.split_summary_display)
    
    # 6. parquet schema 검증 결과를 출력한다.
    print("required_non_model_columns:", preview.required_non_model_columns)
    print("optional_trace_columns:", preview.optional_trace_columns)
    for row in preview.schema_summary.to_dict(orient="records"):
        if row["missing_optional"]:
            print(f"{row['split']}_optional_trace_columns_missing: {row['missing_optional']}")
        print(f"{row['split']}_schema: OK, column_count={row['column_count']}, path={row['path']}")
        
    # 7. 여기까지 통과하면 승인 feature mask와 입력 parquet 계약은 학습 전 기준을 만족한다.
    print("approved feature mask: OK")
    print("out_dir:", OUT_DIR)
    
    # 8. main()이 후속 학습/검증 단계에서 재사용할 핵심 객체만 반환한다.
    return preview.feature_columns, preview.split_summary

print("functions loaded")

functions loaded


# 04_train + validation 실행 함수

In [6]:
def run_train_and_validation(
    paths: ml_00_ml_io.InputPaths,
) -> tuple[ml_00_ml_train.XGBTrainResult, ml_00_ml_val.ValidationResult]:
    """
    train 학습과 validation threshold 선택을 순서대로 실행한다.
    실행 조건
    --------
    - RUN_TRAIN_AND_VALIDATION=True일 때 main()에서 호출된다.
    - SAMPLE_ROWS가 정수이면 sample smoke/debug 실행이다.
    - SAMPLE_ROWS=None이면 full train/validation 실행이다.
    
    처리 순서
    --------
    1. XGBoost 학습 설정 생성
    2. train split으로 모델 학습
    3. val split으로 early stopping 수행
    4. 학습 artifact 저장
    5. 저장된 모델로 validation threshold 선택
    6. validation metric과 threshold artifact 저장
    
    주의
    ----
    - 이 함수는 final test를 실행하지 않는다.
    - validation metric은 모델/threshold 선택용이며 최종 성능 주장이 아니다.
    """
    
    # USE_NOTEBOOK_XGB_PARAMS=True이면 노트북의 XGB_PARAMS로 모듈 기본값을 덮어쓴다.
    # False이면 빈 dict를 넘겨 ml_00_ml_train.XGBTrainConfig의 기본값을 그대로 사용한다.
    xgb_param_overrides = XGB_PARAMS if USE_NOTEBOOK_XGB_PARAMS else {}
    
    # XGBoost 학습 설정 객체를 만든다. 실제 학습 로직은 ml_00_ml_train.train_xgb()에 위임한다.
    train_config = ml_00_ml_train.XGBTrainConfig(
        train_path=paths.train_path,                         # 학습에 사용할 train parquet
        val_path=paths.val_path,                             # early stopping 평가에 사용할 validation parquet
        feature_columns_path=paths.feature_columns_path,     # used_in_ml="TRUE" feature mask CSV
        output_dir=OUT_DIR,                                  # model.pkl, train_summary.json 저장 위치
        project_root=PROJECT_ROOT,                           
        label_col=LABEL_COL,                                 
        sample_rows=SAMPLE_ROWS,                             
        overwrite=OVERWRITE_POLICY["train"],                 
        allow_nan=False,                                      
        seed=SEED,                                           
        **xgb_param_overrides,                               # 노트북 XGB override 또는 빈 dict
    )
    
    # train_xgb() 내부에서 수행되는 주요 작업:
    # - feature CSV 로딩, train/val parquet 로딩
    # - feature/label/split 검증
    # - scale_pos_weight 자동 계산
    # - XGBoost fit
    # - model.pkl, feature_columns.json, train_summary.json, feature_importance.csv 저장
    train_result = ml_00_ml_train.train_xgb(train_config)
    
    # 학습 결과 핵심 artifact와 학습 데이터 규모를 화면에 출력한다.
    print("train_summary_path:", train_result.train_summary_path)
    print("model_path:", train_result.model_path)
    print("train_rows:", train_result.train_rows)
    print("val_rows_for_early_stopping:", train_result.val_rows)
    print("scale_pos_weight:", train_result.scale_pos_weight)
    
    # validation threshold 선택 설정 객체를 만든다.
    # validate_xgb()는 위에서 저장한 model.pkl과 feature_columns.json을 사용한다.
    val_config = ml_00_ml_val.ValidationConfig(
        val_path=paths.val_path,                             # threshold 선택에 사용할 validation parquet
        output_dir=OUT_DIR,                                  # train artifact가 저장된 동일 폴더
        project_root=PROJECT_ROOT,
        label_col=LABEL_COL,
        sample_rows=SAMPLE_ROWS,                             # sample이면 validation도 같은 row 제한 적용
        allow_nan=False,
        overwrite=OVERWRITE_POLICY["val"],                   # 기존 validation artifact 덮어쓰기 여부
        threshold_strategy=THRESHOLD_STRATEGY,               # max_f1 또는 manual
        manual_threshold=MANUAL_THRESHOLD,                   # manual 전략일 때만 사용
    )
    
    # validate_xgb() 내부에서 수행되는 주요 작업:
    # - model.pkl 로드
    # - feature_columns.json 로드
    # - validation parquet 로드
    # - predict_proba()로 positive class probability 계산
    # - validation 기준 threshold 선택
    # - threshold.json, metrics_val.json, confusion_matrix_val.csv 저장
    # - metrics_val.json에 runtime/resource/score profile 기록
    val_result = ml_00_ml_val.validate_xgb(val_config)
    
    print("threshold_path:", val_result.threshold_path)
    print("val_metrics:", val_result.val_metrics["summary"])
    if SAMPLE_ROWS is not None:
        print("주의: sample 결과는 smoke/debug 전용이며 성능 주장에 사용하지 않습니다.")
    return train_result, val_result

# ============================================================

def run_final_test(paths: ml_00_ml_io.InputPaths) -> ml_00_ml_test.TestResult:
    """
    full train/validation이 끝난 뒤 최종 test split 평가를 실행한다.
    실행 조건
    --------
    - RUN_FINAL_TEST=True일 때 main()에서 호출된다.
    - SAMPLE_ROWS=None이어야 한다.
    - 같은 OUT_DIR에 full train/validation artifact가 있어야 한다.
    - model.pkl, feature_columns.json, train_summary.json, threshold.json이 필요하다.
    처리 순서
    --------
    1. sample 기반 final test 차단
    2. test parquet 경로 확인
    3. final test 설정 객체 생성
    4. ml_00_ml_test.test_xgb() 실행
    5. test metric과 confusion matrix 저장
    주의
    ----
    - final test는 모델/feature/threshold 선택이 끝난 뒤 1회만 실행한다.
    - test set에서 threshold를 다시 고르지 않는다.
    - validation에서 저장된 threshold.json을 그대로 사용한다.
    """
    # sample model/threshold로 final test를 실행하면 최종 평가로 볼 수 없으므로 차단한다.
    # full train/validation을 먼저 수행하려면 SAMPLE_ROWS=None으로 설정해야 한다.
    if SAMPLE_ROWS is not None:
        raise ValueError(
            "Final test is blocked because SAMPLE_ROWS is not None. "
            "Set SAMPLE_ROWS=None, rerun full train/validation, then set RUN_FINAL_TEST=True."
        )
        
    # final test에는 test parquet가 반드시 필요하다.
    if paths.test_path is None:
        raise ValueError("ML_INPUT_TEST_PATH is required for final test.")
    
    # final test 설정 객체를 만든다.
    # confirm_final_test=True는 실수로 test 평가를 실행하지 못하게 하는 명시적 안전장치다.
    test_config = ml_00_ml_test.TestConfig(
        test_path=paths.test_path,                           # 최종 평가용 test parquet
        output_dir=OUT_DIR,                                  # full train/validation artifact가 있는 폴더
        project_root=PROJECT_ROOT,
        label_col=LABEL_COL,
        confirm_final_test=True,                             # final test 실행 명시 승인 플래그
        sample_rows=None,                                    # final test는 항상 전체 test split만 사용
        allow_nan=False,
        overwrite=OVERWRITE_POLICY["test"],                  # 기존 test artifact 덮어쓰기 여부
    )
    
    # test_xgb() 내부에서 수행되는 주요 작업:
    # - model.pkl, feature_columns.json, train_summary.json, threshold.json 로드
    # - sampled model/threshold 차단
    # - validation threshold를 그대로 사용
    # - test parquet 로드
    # - metrics_test.json, confusion_matrix_test.csv 저장
    test_result = ml_00_ml_test.test_xgb(test_config)
    
    print("test_metrics_path:", test_result.metrics_path)
    print("test_metrics:", test_result.test_metrics["summary"])
    
    return test_result

print("functions loaded")

functions loaded


# 05_main 실행

In [7]:
def main() -> dict:
    """
    승인 ML 입력 기준으로 입력 검증, 선택적 학습/validation, 선택적 final test를 순서대로 실행한다.
    실행 흐름
    --------
    1. INPUT_PATHS로 train/val/test/feature mask 입력 경로를 고정
    2. 실행 조합 안전성 확인
    3. 입력 파일, feature mask, split_summary, parquet schema 사전 검증
    4. RUN_TRAIN_AND_VALIDATION=True이면 train + validation 실행
    5. RUN_TRAIN_AND_VALIDATION=False이면 학습/validation을 건너뜀
    6. RUN_FINAL_TEST=True이면 같은 OUT_DIR의 full artifact로 final test 실행
    반환
    ----
    - feature_columns: 실제 모델 입력 feature 목록
    - split_summary: split_summary.csv 검증 요약. 기본값에서는 test label 분포를 숨긴다.
    - train_result: 학습 실행 결과. 미실행 시 None
    - val_result: validation 실행 결과. 미실행 시 None
    - test_result: final test 실행 결과. 미실행 시 None
    """
    # 설정 셀에서 한 번 만든 입력 경로 객체를 전체 실행 흐름에서 재사용한다.
    # main() 내부에서 경로를 다시 조립하지 않아야 설정 source of truth가 분산되지 않는다.
    paths = INPUT_PATHS
    
    # sample model 또는 sample threshold로 final test를 실행하는 것을 차단한다.
    # final test는 SAMPLE_ROWS=None인 full train/validation 이후에만 허용한다.
    if RUN_FINAL_TEST and SAMPLE_ROWS is not None:
        raise ValueError(
            "RUN_FINAL_TEST=True requires SAMPLE_ROWS=None. "
            "Final test must use the full train/validation artifacts."
        )
        
    # 학습 전에 입력 계약을 검증한다.
    # 여기서는 학습을 실행하지 않고 아래만 확인한다.
    # - 입력 파일 존재 여부
    # - approve CSV의 모델 입력 feature 목록
    # - split_summary.csv 요약. test label 분포는 승인 플래그가 있을 때만 표시
    # - train/val/test parquet schema
    feature_columns, split_summary = preview_ml_inputs(paths)
    
    # 노트북 실행 결과를 한 객체로 모아둔다.
    # 학습/validation/final test를 실행하지 않은 경우 해당 result는 None으로 남긴다.
    result = {
        "feature_columns": feature_columns,
        "split_summary": split_summary,
        "train_result": None,
        "val_result": None,
        "test_result": None,
    }
    
    # RUN_TRAIN_AND_VALIDATION=True이면 train -> validation 순서로 실행한다.
    # False이면 기존 full artifact를 사용하는 final-test-only 실행도 가능하도록 학습 단계를 건너뛴다.
    if RUN_TRAIN_AND_VALIDATION:
        try:
            train_result, val_result = run_train_and_validation(paths)
        except ImportError as exc:
            raise SystemExit(str(exc))

        # 학습과 validation 결과 객체를 반환 payload에 저장한다.
        result["train_result"] = train_result
        result["val_result"] = val_result
    else:
        print("Train/validation skipped. Set RUN_TRAIN_AND_VALIDATION=True to train.")
    
    # final test는 명시적으로 RUN_FINAL_TEST=True일 때만 실행한다.
    # threshold는 validation에서 저장한 threshold.json을 그대로 사용해야 한다.
    if RUN_FINAL_TEST:
        result["test_result"] = run_final_test(paths)
    else:
        print("Final test skipped. Set SAMPLE_ROWS=None and RUN_FINAL_TEST=True only for final evaluation.")
        
    return result

# 노트북 실행 진입점.
# 설정 셀의 스위치 값에 따라 입력 검증만 하거나, train/validation/final test까지 이어서 수행한다.
MAIN_RESULT = main()

train_path          : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_Xy_train.parquet
val_path            : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_Xy_val.parquet
test_path           : /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_Xy_test.parquet
feature_columns_path: /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_ml_feature_columns_approve.csv
model_feature_columns_path: /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_ml_feature_columns_approve.csv
model_feature_count: 14
모델 입력 feature 최종 확인표


,feature_order,column_name,used_in_ml,feature_group,dtype,leakage_risk,selection_note,target_experiment,used_in_experiments,excluded_reason
0,1,amount__current__log1p,TRUE,amount,Float64,low_current_row_only,NaN,ml_exp00,ml_exp00;ml_exp00_no_time,NaN
1,2,amount__current__value,TRUE,amount,Float64,low_current_row_only,NaN,ml_exp00,ml_exp00;ml_exp00_no_time,NaN
2,3,amount__paid_recv_ratio,TRUE,amount,Float64,low_current_row_only,NaN,candidate,NaN,add_candidate: 기본 False. parquet에 있으면 True로 변경 가능
3,4,amount_received__current__log1p,TRUE,amount,Float64,low_current_row_only,NaN,candidate,NaN,add_candidate: 기본 False. parquet에 있으면 True로 변경 가능
4,5,amount_received__current__value,TRUE,amount,Float64,low_current_row_only,NaN,candidate,NaN,add_candidate: 기본 False. parquet에 있으면 True로 변경 가능
5,6,cat__bank_pair__code,TRUE,categorical_label_code,Int32,low_if_train_only_mapping,candidate (parquet 포함),candidate,NaN,candidate: 성능비교 후 True로 변경 (parquet에 포함됨)
6,7,cat__currency_pair__code,TRUE,categorical_label_code,Int32,low_if_train_only_mapping,candidate (parquet 포함),candidate,NaN,candidate: 성능비교 후 True로 변경 (parquet에 포함됨)
7,8,cat__from_bank__code,TRUE,categorical_label_code,Int32,low_if_train_only_mapping,NaN,ml_exp00,ml_exp00;ml_exp00_no_time,NaN
8,9,cat__payment_currency__code,TRUE,categorical_label_code,Int32,low_if_train_only_mapping,NaN,ml_exp00,ml_exp00;ml_exp00_no_time,NaN
9,10,cat__payment_format__code,TRUE,categorical_label_code,Int32,low_if_train_only_mapping,NaN,ml_exp00,ml_exp00;ml_exp00_no_time,NaN


feature_count: 14
feature_columns_hash: 8a43bb28280a274b837c729ce44cc066226ce66b6db5168c652ea7a66f03d135
feature_columns: ['amount__current__log1p', 'amount__current__value', 'amount__paid_recv_ratio', 'amount_received__current__log1p', 'amount_received__current__value', 'cat__bank_pair__code', 'cat__currency_pair__code', 'cat__from_bank__code', 'cat__payment_currency__code', 'cat__payment_format__code', 'cat__receiving_currency__code', 'cat__to_bank__code', 'time__row__dayofweek', 'time__row__is_weekend']
split_summary_path: /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_split_summary.csv
전처리/feature build split 요약 (test label 분포 숨김)
SHOW_TEST_LABEL_DISTRIBUTION=True일 때만 test label 분포를 표시합니다.


,split,rows,timestamp_min,timestamp_max,label_0_count,label_1_count,positive_rate
0,train,3046861,2022-09-01 00:00:00,2022-09-06 13:35:00,3044564,2297,0.000754
1,val,1015920,2022-09-06 13:36:00,2022-09-08 16:12:00,1014837,1083,0.001066
2,test,1015564,2022-09-08 16:13:00,2022-09-18 16:18:00,[hidden],[hidden],[hidden]


required_non_model_columns: ['label', 'split', 'timestamp', 'tx_id']
optional_trace_columns: ['receiver_account', 'sender_account']
train_schema: OK, column_count=21, path=/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_Xy_train.parquet
val_schema: OK, column_count=21, path=/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_Xy_val.parquet
test_schema: OK, column_count=21, path=/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_Xy_test.parquet
approved feature mask: OK
out_dir: /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour
train_summary_path: /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour/train_summary.json
model_path: /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/ru

# 06_output artifact 상태 확인

In [8]:
input_artifact_paths = {
    "train_path": INPUT_PATHS.train_path,
    "val_path": INPUT_PATHS.val_path,
    "test_path": INPUT_PATHS.test_path,
    "feature_columns_path": INPUT_PATHS.feature_columns_path,
    "split_summary_path": ML_INPUT_SPLIT_SUMMARY_PATH,
}

output_artifact_paths = {
    "model_path": OUT_DIR / "model.pkl",
    "feature_columns_path": OUT_DIR / "feature_columns.json",
    "train_summary_path": OUT_DIR / "train_summary.json",
    "feature_importance_path": OUT_DIR / "feature_importance.csv",
    "threshold_path": OUT_DIR / "threshold.json",
    "metrics_val_path": OUT_DIR / "metrics_val.json",
    "confusion_matrix_val_path": OUT_DIR / "confusion_matrix_val.csv",
    "metrics_test_path": OUT_DIR / "metrics_test.json",
    "confusion_matrix_test_path": OUT_DIR / "confusion_matrix_test.csv",
}

input_artifact_status = pd.DataFrame(
    [artifact_file_status(name, path) for name, path in input_artifact_paths.items() if path is not None]
)
output_artifact_status = pd.DataFrame(
    [artifact_file_status(name, path) for name, path in output_artifact_paths.items()]
)

print("ml_inputs 승인 입력 산출물 상태")
display_status_table(input_artifact_status)
print("ml_outputs 학습/검증/테스트 산출물 상태")
display_status_table(output_artifact_status)
print("후속 실행 기준")
print("MODEL_RUN_ID            =", MODEL_RUN_ID)
print("RUN_TRAIN_AND_VALIDATION =", RUN_TRAIN_AND_VALIDATION)
print("SAMPLE_ROWS              =", SAMPLE_ROWS)
print("RUN_FINAL_TEST           =", RUN_FINAL_TEST)
print("SHOW_TEST_LABEL_DISTRIBUTION =", SHOW_TEST_LABEL_DISTRIBUTION)
print("OUT_DIR                  =", OUT_DIR)
print()
if RUN_TRAIN_AND_VALIDATION:
    print("이번 실행은 train/validation을 수행했습니다.")
else:
    print("이번 실행은 train/validation을 수행하지 않았습니다.")

if SAMPLE_ROWS is not None:
    print("주의: sample 결과는 smoke/debug 전용이며 성능 주장에 사용하지 않습니다.")
else:
    print("이번 실행은 full split 기준입니다.")
print("주의: validation 결과가 안 좋으면 RUN_ID가 아니라 MODEL_RUN_ID만 바꿔 재시도하세요.")
print("주의: final test는 full model과 validation threshold 확정 후 1회만 실행하세요.")


ml_inputs 승인 입력 산출물 상태


,name,exists,is_file,modified_at,path
0,train_path,True,True,2026-05-16 19:58:59.318575,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_Xy_train.parquet
1,val_path,True,True,2026-05-16 19:59:00.081700,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_Xy_val.parquet
2,test_path,True,True,2026-05-16 19:59:00.867917,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_Xy_test.parquet
3,feature_columns_path,True,True,2026-05-18 15:59:29.737201,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_ml_feature_columns_approve.csv
4,split_summary_path,True,True,2026-05-16 19:59:00.883613,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_inputs/ml_00_test/run_00_no_hour/ml_00_test__run_00_no_hour_split_summary.csv


ml_outputs 학습/검증/테스트 산출물 상태


,name,exists,is_file,modified_at,path
0,model_path,True,True,2026-05-18 16:09:42.480074,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour/model.pkl
1,feature_columns_path,True,True,2026-05-18 16:09:42.486073,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour/feature_columns.json
2,train_summary_path,True,True,2026-05-18 16:09:42.528196,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour/train_summary.json
3,feature_importance_path,True,True,2026-05-18 16:09:42.496195,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour/feature_importance.csv
4,threshold_path,True,True,2026-05-18 16:09:44.806596,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour/threshold.json
5,metrics_val_path,True,True,2026-05-18 16:09:44.827787,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour/metrics_val.json
6,confusion_matrix_val_path,True,True,2026-05-18 16:09:44.813595,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour/confusion_matrix_val.csv
7,metrics_test_path,False,False,NaT,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour/metrics_test.json
8,confusion_matrix_test_path,False,False,NaT,/mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour/confusion_matrix_test.csv


후속 실행 기준
MODEL_RUN_ID            = default_00_refactoring_no_hour
RUN_TRAIN_AND_VALIDATION = True
SAMPLE_ROWS              = None
RUN_FINAL_TEST           = False
SHOW_TEST_LABEL_DISTRIBUTION = False
OUT_DIR                  = /mnt/d/AML_git/Graph_AML/ml/ml-00_baseline/ml_outputs/ml_00_test/run_00_no_hour/full/default_00_refactoring_no_hour

이번 실행은 train/validation을 수행했습니다.
이번 실행은 full split 기준입니다.
주의: validation 결과가 안 좋으면 RUN_ID가 아니라 MODEL_RUN_ID만 바꿔 재시도하세요.
주의: final test는 full model과 validation threshold 확정 후 1회만 실행하세요.
